# Demo 02 : create_agent

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · create-agent

We build the same loop from a model and a tool list.

In [2]:
import json
import sqlite3

from langchain.tools import tool

DB = data("demo-world/service.db")


@tool
def bus_info(fleet_no: str) -> str:
    """Look up a Veenhaven bus model and depot by its fleet number."""
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT model, depot_id FROM buses WHERE fleet_no = ?", (fleet_no,)
        ).fetchone()
    return json.dumps(dict(zip(["model", "depot_id"], row))) if row else "Unknown bus"


@tool
def parts_for_model(model: str) -> str:
    """List spare parts, stock and storage depots for a bus model such as CE-12."""
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            "SELECT part_no, name, stock, depot FROM parts WHERE model = ?", (model,)
        ).fetchall()
    return json.dumps([dict(zip(["part_no", "name", "stock", "depot"], row)) for row in rows])


tools = [bus_info, parts_for_model]
question = {"messages": [("human", "Which door actuator fits bus CE12-2021-0117, and is it in stock? Look up the bus first.")]}

We supply the same tools to `create_agent`. The return value is a compiled LangGraph graph, and invocation returns its final state.

In [3]:
from langchain.agents import create_agent

agent = create_agent(llm, tools=tools,
                     system_prompt="Look up the bus first. Use the database for stock. Report unknown records.")
result = agent.invoke(question, {"recursion_limit": 16})
for message in result["messages"]:
    print(message.type, getattr(message, "tool_calls", None) or message.content)
print("graph type:", type(agent).__name__)
print("nodes:", list(agent.get_graph().nodes))
print("tool schema:", bus_info.args)

human Which door actuator fits bus CE12-2021-0117, and is it in stock? Look up the bus first.
ai [{'name': 'bus_info', 'args': {'fleet_no': 'CE12-2021-0117'}, 'id': 'call_B2GljcNEsv02jjU5acA1sRie', 'type': 'tool_call'}]
tool {"model": "CE-12", "depot_id": "D01"}
ai [{'name': 'parts_for_model', 'args': {'model': 'CE-12'}, 'id': 'call_YLPe1Ju8xZP19hwRAq3ZPB2r', 'type': 'tool_call'}]
tool [{"part_no": "CE-TB-01", "name": "traction battery module", "stock": 1, "depot": "Rietpoort"}, {"part_no": "CE-CF-02", "name": "traction cooling fan", "stock": 6, "depot": "Rietpoort"}, {"part_no": "CE-DA-03", "name": "door actuator", "stock": 0, "depot": "Rietpoort"}, {"part_no": "CE-IM-04", "name": "inverter module", "stock": 2, "depot": "Broekdam"}]
ai Bus **CE12-2021-0117** is a **CE-12**, assigned to depot **D01**. The matching door actuator is **CE-DA-03**. The database shows **0 in stock** at **Rietpoort**, so it is not currently available there.
graph type: CompiledStateGraph
nodes: ['__start__',

## b · agent-as-a-node

We put the compiled agent inside a parent graph. Shared message state passes into and out of the agent node.

In [4]:
from langgraph.graph import END, START, MessagesState, StateGraph


class VisitState(MessagesState):
    handover: str


def handover(state: VisitState):
    return {"handover": "Workshop review required. " + str(state["messages"][-1].content)}


parent = StateGraph(VisitState)
parent.add_node("parts_agent", agent)
parent.add_node("handover", handover)
parent.add_edge(START, "parts_agent")
parent.add_edge("parts_agent", "handover")
parent.add_edge("handover", END)
out = parent.compile().invoke(question)
print("handover:", out["handover"])

handover: Workshop review required. Bus **CE12-2021-0117** is a **CE-12**, assigned to depot **D01**. The matching door actuator is **CE-DA-03**. The parts database lists **0 in stock** at **Rietpoort**.


## c · custom-agent-state

We extend `AgentState` with a depot and a lookup counter. A model hook adds the depot to the prompt, and a tool reads state through `ToolRuntime`. The runtime argument stays out of the tool schema.

In [5]:
from langchain.agents import AgentState
from langchain.agents.middleware import after_model, dynamic_prompt
from langchain.tools import ToolRuntime
from typing_extensions import NotRequired


class DepotState(AgentState):
    depot: str
    model_calls: NotRequired[int]


@dynamic_prompt
def depot_prompt(request):
    return f"You assist the workshop at {request.state['depot']}. Use depot_parts for stock."


@after_model(state_schema=DepotState)
def count_calls(state, runtime):
    return {"model_calls": state.get("model_calls", 0) + 1}


@tool
def depot_parts(runtime: ToolRuntime) -> str:
    """List spare parts and stock at the current workshop depot."""
    with sqlite3.connect(DB) as conn:
        rows = conn.execute("SELECT model, part_no, name, stock FROM parts WHERE depot = ?",
                            (runtime.state["depot"],)).fetchall()
    return json.dumps([dict(zip(["model", "part_no", "name", "stock"], row)) for row in rows])


depot_agent = create_agent(llm, tools=[depot_parts], state_schema=DepotState,
                           middleware=[depot_prompt, count_calls])
out = depot_agent.invoke({"depot": "Rietpoort", "messages": [
    ("human", "Is a CE-12 door actuator available at my workshop? Use the tool.")]})
print("depot:", out["depot"])
print("model calls:", out["model_calls"])
print("tool schema:", depot_parts.args)
print("answer:", out["messages"][-1].content)

depot: Rietpoort
model calls: 2
tool schema: {}
answer: No. The CE-12 door actuator (part CE-DA-03) is out of stock at your workshop.


## Things to try

Optional : change the bus or part identifier and run the example again.